In [ ]:
import json
import cv2
import sys
import torch
sys.path.append("/Users/abhinavarora/Desktop/CadenceCV/src")
import yolo_model_loader
with open("/Users/abhinavarora/Desktop/CadenceCV/data/strikefoot_data.json", "r") as file:
    data = json.load(file)

model = yolo_model_loader.load_model("/Users/abhinavarora/Desktop/CadenceCV/models/21_keypoints.pt")

base_video_dir = "/Users/abhinavarora/Desktop/CadenceCV/Videos"
video_extensions = [".mp4", ".MOV"]
for frame in data:
    for ext in video_extensions:
        try:
            dir = base_video_dir + "/" + frame["video"] + ext
            cap = cv2.VideoCapture(dir)
            cap.set(cv2.CAP_PROP_POS_FRAMES, frame["frame"])
            ret, img = cap.read()
            source = model.predict(img, conf = 0.3)

            for result in source:
                kpts = result.keypoints.xy[0]

            # Extract relevant keypoints
            left_big_toe = kpts[17]
            right_big_toe = kpts[18]
            left_heel = kpts[19]
            right_heel = kpts[20]

            left_ankle = kpts[15]
            right_ankle = kpts[16]

            # Foot-ground angle: angle of foot vector (heel to big toe) relative to horizontal
            # Negative y because image coordinates are flipped (y increases downward)
            left_foot_vector = left_big_toe - left_heel
            right_foot_vector = right_big_toe - right_heel

            left_foot_angle = torch.atan2(-left_foot_vector[1], left_foot_vector[0])
            right_foot_angle = torch.atan2(-right_foot_vector[1], right_foot_vector[0])

            # Convert to degrees
            left_foot_angle_deg = torch.rad2deg(left_foot_angle).item()
            right_foot_angle_deg = torch.rad2deg(right_foot_angle).item()

            frame["left_foot_angle"] = left_foot_angle_deg
            frame["right_foot_angle"] = right_foot_angle_deg

            print(left_foot_angle_deg)
            print(right_foot_angle_deg)
            break

        except Exception:
            continue
            




In [12]:
for strike in data:
    if strike["side"] == "R":
        strike["foot_angle"] = strike["right_foot_angle"]
    else:
        strike["foot_angle"] = strike["left_foot_angle"]

In [14]:
with open("/Users/abhinavarora/Desktop/CadenceCV/data/strikefoot_data.json", "w") as file:
    json.dump(data, file, indent=4)